# Цели исследования:
Коронавирус застал мир врасплох, изменив привычный порядок вещей. В свободное время жители городов больше не выходят на улицу, не посещают кафе и торговые центры. Зато стало больше времени для книг. Это заметили стартаперы — и бросились создавать приложения для тех, кто любит читать.

Ваша компания решила быть на волне и купила крупный сервис для чтения книг по подписке. Ваша первая задача как аналитика — проанализировать базу данных. В ней — информация о книгах, издательствах, авторах, а также пользовательские обзоры книг. Эти данные помогут сформулировать ценностное предложение для нового продукта.

# Описание данных:
__Таблица__ _books_<br>
Содержит данные о книгах:<br>
book_id — идентификатор книги;<br>
author_id — идентификатор автора;<br>
title — название книги;<br>
num_pages — количество страниц;<br>
publication_date — дата публикации книги;<br>
publisher_id — идентификатор издателя.<br><br>
__Таблица__ _Таблица authors_<br>
Содержит данные об авторах:<br>
author_id — идентификатор автора;<br>
author — имя автора.<br><br>
__Таблица__ _publishers_<br>
Содержит данные об издательствах:<br>
publisher_id — идентификатор издательства;<br>
publisher — название издательства;<br><br>
__Таблица__ _ratings_<br>
Содержит данные о пользовательских оценках книг:<br>
rating_id — идентификатор оценки;<br>
book_id — идентификатор книги;<br>
username — имя пользователя, оставившего оценку;<br>
rating — оценка книги.<br><br>
__Таблица__ _reviews_<br>
Содержит данные о пользовательских обзорах на книги:<br>
review_id — идентификатор обзора;<br>
book_id — идентификатор книги;<br>
username — имя пользователя, написавшего обзор;<br>
text — текст обзора.

# Задания
* Посчитайте, сколько книг вышло после 1 января 2000 года;
* Для каждой книги посчитайте количество обзоров и среднюю оценку;
* Определите издательство, которое выпустило наибольшее число книг толще 50 страниц — так вы исключите из анализа брошюры;
* Определите автора с самой высокой средней оценкой книг — учитывайте только книги с 50 и более оценками;
* Посчитайте среднее количество обзоров от пользователей, которые поставили больше 48 оценок.

In [1]:
# импортируем библиотеки
import pandas as pd
import sqlalchemy as sa

# устанавливаем параметры
db_config = {
    'user': 'praktikum_student', # имя пользователя
    'pwd': 'Sdf4$2;d-d30pp', # пароль
    'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
    'port': 6432, # порт подключения
    'db': 'data-analyst-final-project-db' # название базы данных
    }
connection_string = 'postgresql://{user}:{pwd}@{host}:{port}/{db}'.format(**db_config)

# сохраняем коннектор
engine = sa.create_engine(connection_string, connect_args={'sslmode':'require'})



In [2]:
# чтобы выполнить SQL-запрос, пишем функцию с использованием Pandas
def get_sql_data(query:str, engine:sa.engine.base.Engine=engine) -> pd.DataFrame:
    '''Открываем соединение, получаем данные из sql, закрываем соединение'''
    with engine.connect() as con:
        return pd.read_sql(sql=sa.text(query), con = con)



Рассмотрим детальнее таблицы, из которых будем вытаскивать информацию по заданиям. Выведем первые 5 записей, а также информацию о том сколько всего записей в таблице с использованием:

In [3]:
tables_list = ['books', 'authors', 'publishers', 'ratings', 'reviews']
for i in tables_list:
    query_1 = str("SELECT * FROM " + i + " LIMIT 5")
    query_2 = str("SELECT COUNT(*) FROM " + i)
    print(f'Информация по таблице {i}')
    print('Первые 5 строк:')
    display(get_sql_data(query_1))
    print(f'В таблице {i} ровно {str(get_sql_data(query_2)).split()[2]} строк') # альтернатива {get_sql_data(query_2).iloc[0, 0]}
    print('\n'*2)

Информация по таблице books
Первые 5 строк:


,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268


В таблице books ровно 1000 строк



Информация по таблице authors
Первые 5 строк:


,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd


В таблице authors ровно 636 строк



Информация по таблице publishers
Первые 5 строк:


...

В таблице publishers ровно 340 строк



Информация по таблице ratings
Первые 5 строк:


,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2


В таблице ratings ровно 6456 строк



Информация по таблице reviews
Первые 5 строк:


,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


В таблице reviews ровно 2793 строк





Приступим к заданиям:

In [4]:
#Посчитайте, сколько книг вышло после 1 января 2000 года;
query = """

SELECT 
    COUNT(*) 
FROM 
    books 
WHERE publication_date > '2000-01-01'

    """
print(f'Всего вышло {str(get_sql_data(query)).split()[2]} книг после 1 января 2000 года')

Всего вышло 819 книг после 1 января 2000 года


In [5]:
#Для каждой книги посчитайте количество обзоров и среднюю оценку:
query = """

SELECT 
    b.title AS "Название книги",
    COUNT(DISTINCT re.review_id) AS "Количество обзоров", 
    AVG(ra.rating) AS "Средний рейтинг"
FROM 
    books AS b
    LEFT JOIN reviews as re on b.book_id=re.book_id
    JOIN ratings AS ra on b.book_id=ra.book_id

GROUP BY b.book_id -- долго не понимал, что с "Мемуарами гейши" делать. 
ORDER BY "Средний рейтинг" DESC

    """

get_sql_data(query)

,Название книги,Количество обзоров,Средний рейтинг
0,Arrows of the Queen (Heralds of Valdemar #1),2,5.00
1,The Walking Dead Book One (The Walking Dead #...,2,5.00
2,Light in August,2,5.00
3,Wherever You Go There You Are: Mindfulness Me...,2,5.00
4,Captivating: Unveiling the Mystery of a Woman'...,2,5.00
...,...,...,...
995,The World Is Flat: A Brief History of the Twen...,3,2.25
996,His Excellency: George Washington,2,2.00
997,Drowning Ruth,3,2.00
998,Junky,2,2.00


In [6]:
#Определите издательство, которое выпустило наибольшее число книг толще 50 страниц — так вы исключите из анализа брошюры:
query = """

SELECT 
    publisher AS "Издательство", 
    count(*) AS "Количество книг"
FROM 
    books AS b
    JOIN publishers AS p 
        ON b.publisher_id=p.publisher_id
WHERE num_pages > 50  
GROUP BY publisher
ORDER BY count(*) DESC
LIMIT 1
 
    """

print(f'Больше всего книг толще 50 страниц выпустило издательство "{get_sql_data(query).iloc[0,0]}".')
print(f'Количество выпущенных книг - {get_sql_data(query).iloc[0,1]}')

Больше всего книг толще 50 страниц выпустило издательство "Penguin Books".
Количество выпущенных книг - 42


In [7]:
#Определите автора с самой высокой средней оценкой книг — учитывайте только книги с 50 и более оценками:
query = """
SELECT 
    a.author AS "Автор", 
    AVG(r.rating) AS "Средний рейтинг"
FROM 
    books AS b
    JOIN authors AS a 
        ON a.author_id=b.author_id
    RIGHT JOIN ratings AS r 
        ON b.book_id=r.book_id
WHERE b.book_id IN (SELECT 
                        b.book_id
                    FROM 
                        books AS b
                        RIGHT JOIN ratings AS r 
                            ON b.book_id=r.book_id
                    GROUP BY b.book_id
                    HAVING COUNT(*) >= 50
                    ) 
GROUP BY a.author
ORDER BY "Средний рейтинг" DESC
LIMIT 1
    """

print(f'Автор с самой высокой средней оценкой книг - "{get_sql_data(query).iloc[0, 0]}, {get_sql_data(query).iloc[0, 1]}" (учитывались только книги, у которых было от 50 оценок)')

Автор с самой высокой средней оценкой книг - "J.K. Rowling/Mary GrandPré, 4.287096774193548" (учитывались только книги, у которых было от 50 оценок)


In [8]:
#Посчитайте среднее количество обзоров от пользователей, которые поставили больше 48 оценок:
query = """
WITH t AS (SELECT
    username,
    COUNT(*) AS "Количество обзоров"
FROM 
    reviews AS r
WHERE r.username IN 
    (SELECT 
            username
        FROM 
            ratings AS r
        GROUP BY username
        HAVING count(*) >48)
    GROUP BY username)

SELECT AVG("Количество обзоров")::int FROM t
    """


print(f'Cреднее количество обзоров от пользователей, которые поставили больше 48 оценок - {str(get_sql_data(query)).split()[2]}')

Cреднее количество обзоров от пользователей, которые поставили больше 48 оценок - 24


# Выводы: 
__По таблицам:__
* В таблице books 1000 записей
* В таблице authors 636 записей
* В таблице publishers 340 записей
* В таблице ratings 6456 записей
* В таблице reviews 2793 записей

Прочая информация дана в описании данных.

__По заданиям:__
1. Количество книг, выпущенных после 1 января 2000 года - 819.
2. По каждой книге посчитано количество обзоров и средняя оценка (выведена таблица).
3. Больше всего книг толще 50 страниц выпустило издательство "Penguin Books". Количество выпущенных книг - 42 штуки.
4. Автор с самой высокой средней оценкой книг - "J.K. Rowling/Mary GrandPré" (учитывались только книги, у которых было от 50 оценок)
5. Cреднее количество обзоров от пользователей, которые поставили больше 48 оценок - 24